In [1]:
# Задание 2

class Image:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.data = []
        
    def get_image(self):
        return self.data
    
    def get_width(self):
        return self.width
    
    def get_height(self):
        return self.height
        
        
class BinaryImage(Image):
    def __init__(self, width, height, bin_image):
        super().__init__(width, height)
        self.data = bin_image
        
class MonochromeImage(Image):
    def __init__(self, width, height, mono_image):
        super().__init__(width, height)
        self.data = mono_image
        
class ColouredImage(Image):
    def __init__(self, width, height, coloured_image):
        super().__init__(width, height)
        self.data = coloured_image
        
bn = BinaryImage(1, 1, [1])
bn.get_image()

mn = MonochromeImage(1, 1, [2])
mn.get_image()

mn = ColouredImage(1, 1, [3])
mn.get_image()


[3]

In [3]:
# Задание 3
import numpy as np
import cv2


class Converter:
    def monochrome_to_monochrome(src_img, trg_img):
        src = src_img.astype(np.float32)
        trg = trg_img.astype(np.float32)
        sm, ss = src.mean(), src.std()
        tm, ts = trg.mean(), trg.std()
        if ss == 0:
            ss = 1
        out = (src - sm) * (ts / ss) + tm
        return np.clip(out, 0, 255).astype(np.uint8)
    
    
    def coloured_to_coloured(src_img, trg_img):
        src_lab = cv2.cvtColor(src_img, cv2.COLOR_BGR2LAB).astype("float32")
        trg_lab = cv2.cvtColor(trg_img, cv2.COLOR_BGR2LAB).astype("float32")
        src_mean, src_std = cv2.meanStdDev(src_lab)
        trg_mean, trg_std = cv2.meanStdDev(trg_lab)
        src_mean, src_std = src_mean.flatten(), src_std.flatten()
        trg_mean, trg_std = trg_mean.flatten(), trg_std.flatten()
        src_std[src_std == 0] = 1
        out = np.zeros(src_lab.shape())
        for i in range(3):
            out[:, :, i] = (src_lab[:, :, i] - src_mean[i]) * (trg_std[i] / src_std[i]) + trg_mean[i]
        out = np.clip(out, 0, 255).astype(np.uint8)
        return cv2.cvtColor(out, cv2.COLOR_LAB2BGR)
    
    def binary_to_binary(img):
        return img
    
    def coloured_to_monochrome(img):
        return ((img[:, :, 0].astype(np.float32) + img[:, :, 1].astype(np.float32) + img[:, :, 2].astype(np.float32)) / 3.0).astype(np.uint8)
    
    def monochrome_to_coloured(img, palette=None):
        if palette is None:
            palette = cv2.applyColorMap(np.arange(256, dtype=np.uint8).reshape(256, 1), cv2.COLORMAP_JET).reshape(256, 3)
        return palette[img]

    
    def monochrome_to_binary(img, threshold=None):
        if threshold is None:
            _, out = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            _, out = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
        return out
    
    def binary_to_monochrome(img):
        inv = cv2.bitwise_not(img)
        dist = cv2.distanceTransform(inv, cv2.DIST_L2, 5)
        if dist.max() > 0:
            dist = dist / dist.max() * 255
        return dist.astype(np.uint8)
    
    def coloured_to_binary(img, threshold=None):
        mono = Converter.coloured_to_monochrome(img)
        return Converter.monochrome_to_binary(mono, threshold)

    def binary_to_coloured(img, palette=None):
        mono = Converter.binary_to_monochrome(img)
        return Converter.monochrome_to_coloured(mono, palette)